<a href="https://colab.research.google.com/github/JuCaRiCo/202602-AnaliticaDescriptivaPredictiva/blob/main/Sesion08_Bloque3_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 8 · Parte 3 de 3 — Web scraping

Este notebook es autoexplicativo: cada tema incluye su **definición**, la **sintaxis** que se usa,
**para qué sirve** cada elemento, y un **ejemplo ejecutable**.

Tercera y última parte de la Sesión 8. Con esta parte se cierra el Tema 1.

### ¿Qué es y cuándo se usa?

> **Definición:** El **web scraping** es la extracción de datos directamente del código de una página web.

**Para qué sirve:** es el último recurso, no el primero.

| Situación | Método preferente |
|---|---|
| Existe un archivo descargable oficial | Descargarlo (Parte 1) |
| Existe una API documentada | Consultarla (Parte 2) |
| Los datos sólo están en una página HTML, y extraerlos está permitido | Web scraping |

Una página fue diseñada para que la vea una persona, no un programa — su estructura puede cambiar sin
aviso. Por eso el scraping suele ser más frágil que una API documentada.

### HTML: lo mínimo necesario

> **Definición:** **HTML** es el lenguaje de estructura de una página web: texto organizado en
> **etiquetas** (`<p>`, `<div>`, `<table>`, `<li>`) que pueden anidarse unas dentro de otras — el mismo
> principio que una lista de diccionarios ya trabajada desde Sesión 4.

**Sintaxis de una etiqueta:** `<etiqueta atributo="valor">contenido</etiqueta>`. Los atributos `class` e
`id` sirven para identificar una etiqueta específica entre muchas del mismo tipo.

**Para qué sirve entender esto:** saber que el dato que se busca vive dentro de alguna etiqueta específica
es lo que permite indicarle a Python dónde buscar.


HTML organiza contenido mediante etiquetas:

```html
<table>              <!-- tabla -->
  <tr>               <!-- fila -->
    <th>Ciudad</th>  <!-- encabezado -->
    <td>Juárez</td>  <!-- celda -->
  </tr>
</table>
```

Otras etiquetas comunes son `<h1>` para encabezados, `<p>` para párrafos, `<a>` para enlaces y `<div>` para contenedores.

### Extraer tablas con `pd.read_html()`

> **Definición:** Cuando una página HTML ya contiene una etiqueta `<table>`, `pandas` puede extraerla
> directamente, sin necesitar BeautifulSoup.

**Sintaxis:**

```python
tablas = pd.read_html(StringIO(html))
```

**Para qué sirve:** `read_html()` regresa una **lista de DataFrames** — no uno solo — porque una página
puede contener varias tablas. `StringIO` envuelve el texto HTML para que `read_html()` lo trate como si
fuera un archivo, que es lo que esta función espera recibir.

El siguiente ejemplo reconstruye como página HTML la misma tabla `estudiantes` ya usada en la Parte 1 —
mismos datos, ahora llegando por una vía distinta.

**Por qué se empieza con HTML controlado, en vez de una página externa real:** al practicar, conviene
primero entender el procedimiento con una página que no cambia, antes de depender de un sitio externo
cuya estructura puede modificarse sin aviso. Esta misma tabla, reescrita como HTML, es una **página
reproducible**: siempre da el mismo resultado, sin importar cuándo se ejecute la celda. Más adelante, en
el ejemplo con GitHub, se aplica el mismo procedimiento sobre una página real.

In [1]:
import pandas as pd
from io import StringIO

# HTML de práctica: una tabla con los mismos datos ya usados en la Parte 1
html_estudiantes = """
<html>
  <body>
    <h1>Calificaciones</h1>
    <table>
      <thead>
        <tr><th>nombre</th><th>calificacion</th></tr>
      </thead>
      <tbody>
        <tr><td>Ana</td><td>92</td></tr>
        <tr><td>Luis</td><td>78</td></tr>
        <tr><td>Marco</td><td>85</td></tr>
        <tr><td>Sofia</td><td>96</td></tr>
      </tbody>
    </table>
  </body>
</html>
"""

# read_html busca todas las etiquetas <table> y regresa una lista de DataFrames, una por tabla encontrada
tablas = pd.read_html(StringIO(html_estudiantes))

print("Número de tablas encontradas:", len(tablas))
print("Tipo del resultado:", type(tablas))

Número de tablas encontradas: 1
Tipo del resultado: <class 'list'>


**Antes de ejecutar la siguiente celda, escribe tu predicción:**

Si la página tuviera dos tablas en vez de una, ¿qué riesgo hay en asumir que la tabla que interesa siempre
está en la posición `tablas[0]`?

In [2]:
# Se selecciona la primera (y aquí, única) tabla de la lista
df_estudiantes_html = tablas[0]
df_estudiantes_html

,nombre,calificacion
0,Ana,92
1,Luis,78
2,Marco,85
3,Sofia,96


Cuando una página tiene varias tablas, seleccionar por posición (`tablas[0]`) es frágil — si el
orden cambia, se toma la tabla equivocada sin que haya ningún error visible. El parámetro `match` busca la
tabla por un texto que debe contener, lo cual es más explícito:

In [3]:
# match busca, entre todas las tablas de la página, la que contiene el texto indicado
tablas_filtradas = pd.read_html(StringIO(html_estudiantes), match="calificacion")

tablas_filtradas[0]

,nombre,calificacion
0,Ana,92
1,Luis,78
2,Marco,85
3,Sofia,96


### BeautifulSoup

> **Definición:** **BeautifulSoup** es una librería de Python que convierte el texto HTML en un objeto
> que se puede recorrer y filtrar. A diferencia de `read_html()`, no está limitada a tablas: sirve para
> cualquier etiqueta (encabezados, párrafos, enlaces, tarjetas).

**Sintaxis:**

```python
from bs4 import BeautifulSoup
sopa = BeautifulSoup(html, "html.parser")

sopa.find("etiqueta")              # primera coincidencia
sopa.find(id="algun_id")           # por atributo id
sopa.find_all("etiqueta")          # todas las coincidencias, como lista
```

**Para qué sirve cada método:** `find()` regresa un solo elemento (o `None` si no existe); `find_all()`
regresa todos los elementos que coinciden, para recorrerlos con un bucle.

**Nota:** BeautifulSoup también acepta selectores CSS con `sopa.select("selector")` (por ejemplo,
`sopa.select(".aviso")` para todos los elementos de la clase `aviso`), como alternativa a `find_all()`.
Este notebook usa `find()`/`find_all()` de forma consistente, pero es útil reconocer `select()` si aparece
en código de otras personas.

### Antes de escribir código: la lista de comprobación

> **Definición:** Antes de extraer datos de cualquier sitio real, se revisan varios puntos — y se
> revisan **antes** de escribir una sola línea de código, no después:

1. ¿Existe una descarga oficial o una API? (Si existe, se prefiere sobre el scraping.)
2. ¿Las condiciones de uso del sitio permiten la extracción?
3. ¿El archivo `robots.txt` establece restricciones técnicas?
4. ¿Se trata de información pública, no de datos personales?
5. ¿La frecuencia de las solicitudes es razonable?
6. ¿Se va a registrar la URL, la fecha y el método usados?

`robots.txt` es una señal técnica dirigida a rastreadores automáticos; no sustituye la revisión de
términos de servicio ni de licencias — son dos verificaciones distintas, ambas necesarias.

El ejemplo de esta sección extrae información pública de una página de GitHub
(`github.com/pandas-dev/pandas`). El archivo `robots.txt` de GitHub permite explícitamente recorrer la
página principal de un repositorio — sólo restringe rutas específicas como `/forks`, `/stargazers` o
`/commits/`, que aquí no se usan.

## Obtener una página con `requests`

Para una página externa:

```python
respuesta = requests.get(url, headers=cabeceras, timeout=30)
respuesta.raise_for_status()
html = respuesta.text
```

- `headers` permite identificar de manera básica el cliente.
- `timeout` evita esperar indefinidamente.
- `raise_for_status()` detecta estados HTTP no exitosos.

Esta celda está desactivada para evitar depender de una página cambiante. Actívala únicamente con una URL pública cuya extracción hayas revisado.

In [4]:
import requests

# Se consulta el robots.txt del sitio ANTES de escribir cualquier código de extracción
robots = requests.get("https://github.com/robots.txt")

print(robots.status_code)
print(robots.text[:300])  # primeras líneas, para confirmar que la página principal no está restringida

200
# If you would like to crawl GitHub contact us via https://support.github.com?tags=dotcom-robots
# We also provide an extensive API: https://docs.github.com
User-agent: bingbot
Disallow: /ekansa/Open-Context-Data
Disallow: /ekansa/opencontext-*
Disallow: /account-login
Disallow: */tarball/
Disallow:


### Ejemplo real: extraer información de una página de GitHub

Con el permiso confirmado, se descarga la página y se convierte en un objeto que se puede recorrer.

In [5]:
from bs4 import BeautifulSoup

url_repositorio = "https://github.com/pandas-dev/pandas"

try:
    # timeout evita esperar indefinidamente; raise_for_status() convierte un status_code de error en excepción
    respuesta = requests.get(url_repositorio, timeout=30)
    respuesta.raise_for_status()
    html = respuesta.text
    origen_html = "github.com (consulta en línea)"

except requests.RequestException as error:
    # Si no hay conexión a GitHub (por ejemplo, en un entorno con acceso a internet restringido),
    # se usa una página de respaldo con la misma estructura mínima, para poder seguir la práctica.
    print("No fue posible consultar la página:", error)
    print("Se usará una página de respaldo, claramente marcada como tal.")
    origen_html = "Página de respaldo (no es una consulta en línea)"
    html = """
    <html><head>
    <title>GitHub - pandas-dev/pandas: Flexible data analysis library</title>
    <meta property="og:description" content="Flexible and powerful data analysis library for Python">
    <meta property="og:site_name" content="GitHub">
    </head></html>
    """

print("Origen utilizado:", origen_html)

# Se convierte el HTML descargado en un objeto que BeautifulSoup puede recorrer
sopa = BeautifulSoup(html, "html.parser")

# find() regresa el primer elemento que coincide, o None si no existe
titulo = sopa.find("title").text
descripcion = sopa.find("meta", property="og:description")

print(titulo)
print(descripcion["content"] if descripcion else None)

Origen utilizado: github.com (consulta en línea)
GitHub - pandas-dev/pandas: Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more · GitHub
Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more - pandas-dev/pandas


**Por qué se usó `<title>` y `<meta property="og:description">`:** son etiquetas presentes en casi
cualquier página web moderna y rara vez cambian — a diferencia de las clases CSS internas del sitio
(usadas para darle estilo visual), que sí cambian con frecuencia y romperían el scraping de un día para
otro.

**Antes de ejecutar la siguiente celda, escribe tu predicción:**

¿Qué pasa si se busca una etiqueta que no existe en la página, por ejemplo `sopa.find("h7")`?

In [6]:
# Esta celda produce un error intencionalmente
etiqueta_inexistente = sopa.find("h7")
print(etiqueta_inexistente.text)

AttributeError: 'NoneType' object has no attribute 'text'

El error es `AttributeError`: `find()` no lanza un error cuando no encuentra la etiqueta, regresa
`None`. El error ocurre después, al intentar leer `.text` de `None`. Por eso siempre conviene revisar si
el resultado de `find()` existe antes de usar `.text`.

### Extraer varios elementos con `find_all()`

La página incluye varias etiquetas `<meta property="og:...">` con información sobre el repositorio.
`find_all()` las recupera todas a la vez, para estructurarlas igual que cualquier lista de diccionarios
trabajada desde Sesión 4.

In [7]:
# property=lambda... selecciona todas las etiquetas <meta> cuyo atributo property empiece con "og:"
metas_og = sopa.find_all("meta", property=lambda valor: valor and valor.startswith("og:"))

info_pagina = []
for etiqueta in metas_og:
    # .get("atributo") no lanza error si el atributo no existe — regresa None en ese caso
    info_pagina.append({
        "propiedad": etiqueta.get("property"),
        "contenido": etiqueta.get("content"),
    })

info_pagina

[{'propiedad': 'og:image',
  'contenido': 'https://opengraph.githubassets.com/f993213ba1e633d1c0d908c89b503ba12752f089fa408a20c0588972b465cadd/pandas-dev/pandas'},
 {'propiedad': 'og:image:alt',
  'contenido': 'Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more - pandas-dev/pandas'},
 {'propiedad': 'og:image:width', 'contenido': '1200'},
 {'propiedad': 'og:image:height', 'contenido': '600'},
 {'propiedad': 'og:site_name', 'contenido': 'GitHub'},
 {'propiedad': 'og:type', 'contenido': 'object'},
 {'propiedad': 'og:title',
  'contenido': 'GitHub - pandas-dev/pandas: Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more'},
 {'propiedad': 'og:url', 'contenido': 'https://github.com/pandas-dev/pandas'},
 {'propiedad': 'og:description',
  'contenido': 

### Ficha de procedencia y guardado

Igual que con una API o un dataset descargado, una extracción por scraping se documenta: de dónde vino,
qué se extrajo, y cuándo.

In [8]:
from datetime import datetime, timezone

ficha_scraping = {
    "origen": url_repositorio,
    "titulo": titulo,
    "fecha_consulta_utc": datetime.now(timezone.utc).isoformat(),
    "metodo": "BeautifulSoup (find/find_all) y pandas.read_html",
    "limitacion": "La estructura HTML del sitio puede cambiar sin aviso",
}

ficha_scraping

{'origen': 'https://github.com/pandas-dev/pandas',
 'titulo': 'GitHub - pandas-dev/pandas: Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more · GitHub',
 'fecha_consulta_utc': '2026-09-05T13:51:34.468382+00:00',
 'metodo': 'BeautifulSoup (find/find_all) y pandas.read_html',
 'limitacion': 'La estructura HTML del sitio puede cambiar sin aviso'}

In [9]:
# index=False evita que el índice del DataFrame se guarde como una columna adicional
df_estudiantes_html.to_csv("estudiantes_desde_html.csv", index=False)

print("Archivo guardado")

Archivo guardado


### Por qué puede fallar el scraping

Antes de depender de un scraping para un proyecto, conviene anticipar por qué podría dejar de funcionar:

- El contenido se genera con JavaScript **después** de cargar la página (esta técnica no lo captura).
- La información se ve como tabla, pero no usa la etiqueta `<table>` — `read_html()` no la encontraría.
- Cambian las clases, los `id`, o la jerarquía del HTML de un día para otro.
- El servidor bloquea solicitudes que detecta como automatizadas.
- El contenido requiere haber iniciado sesión.
- La página cambia de dirección o deja de existir.
- Los datos se extraen, pero de forma incompleta o mal interpretada.

Herramientas como Selenium (que controla un navegador completo) existen para el primer caso — contenido
generado con JavaScript — pero están fuera del alcance de este curso: añaden complejidad considerable y no
resuelven, por sí solas, las restricciones éticas o legales ya vistas.

## Cierre de Unidad 1

Tres caminos distintos —base de datos/repositorio, API, web scraping— y todos terminan en el mismo lugar:
un `DataFrame` listo para analizar. Con esto se cierra el Tema 1. La Sesión 9 abre el Tema 2 con
estadística descriptiva.

## Fuera de alcance de esta parte

- Scraping de páginas que cargan contenido con JavaScript
- Selenium u otras herramientas que controlan un navegador completo

## Glosario de esta parte

| Término | Significado |
|---|---|
| HTML | Lenguaje de estructura de una página web, basado en etiquetas anidadas |
| `pd.read_html()` | Extrae directamente las tablas `<table>` de un HTML, como lista de DataFrames |
| BeautifulSoup | Librería de Python para recorrer y filtrar contenido HTML, más allá de tablas |
| `robots.txt` | Archivo que indica qué partes de un sitio se permite recorrer automáticamente |
| Meta tag | Etiqueta HTML con metadatos de la página (título, descripción); suele ser más estable que las clases de estilo visual |
| Ficha de procedencia | Registro de qué se extrajo, de dónde, y cuándo |
| Web scraping | Extracción de datos directamente del código de una página web |